# CVAE-Xception — Full Retraining on 100% of the Dataset

**Purpose**: Retrain the conditional VAE with an Xception encoder from scratch on the full magnetic-domain dataset (169,671 images).
**Inputs**: `dataset_unificado_v2.npz` (keys `img`, `params`, `labels`), shared `metrics.py` (Kaggle dataset `carloscanamejoy/physicalmetrics`).
**Outputs**: `OUTPUT_DIR/checkpoints/cvae_xception_best.h5` (best checkpoint, weights), `cvae_xception_best_weights.h5`, `cvae_xception_param_scaler.pkl`, training-history JSON, and publication figures in `OUTPUT_DIR/cvae_xception_training/`.
**Execution environment**: Kaggle / Google Colab (run only one setup cell).
**Dependencies**: tensorflow, scikit-learn, scikit-image, matplotlib, seaborn, scipy, tqdm, numpy

Architecture summary:
- Encoder backbone: Xception pretrained on ImageNet, blocks 1–10 frozen, blocks 11–14 trainable.
- Posterior network: GAP(2048) + condition embedding → Dense(512) → Dense(256) → [μ_q, log σ²_q] (128 units each).
- Prior network p(z|θ): condition embedding (32) → Dense(64) → Dense(128) → [μ_p, log σ²_p] (128 units each).
- Decoder: latent z (128) ⊕ parameter embedding (64) = 192-dim input → transposed convolutions 5×5 → 10×10 → 20×20 → 40×40 → 2 residual blocks → tanh output, top-left-cropped to 39×39 inside the model (before any loss or metric computation).
- Loss: `L = (1 − SSIM) + 0.05·L1 + β·KLD(q(z|x,θ) ‖ p(z|θ))`, with β ramped linearly from 1e-6 to 0.18 over 12 epochs, then held constant (40 epochs total).

> Implementation note: the prior heads use 128 units (not 64) so that the KL divergence
> between q(z|x,θ) and p(z|θ) is defined over matching dimensions. The 192-dim decoder
> input corresponds to z (128) concatenated with the 64-dim parameter embedding.

## Kaggle Environment

In [ ]:
# ============================================================
# ENVIRONMENT SETUP — KAGGLE
# Run this cell only when executing on Kaggle.
# All datasets (including metrics.py from the physicalmetrics
# dataset) are pre-mounted read-only under:
#   /kaggle/input/datasets/carloscanamejoy/
# Nothing to download — paths are defined in the Shared
# Configuration cell below.
# ============================================================
import os, sys
print("Kaggle environment ready:", os.path.exists("/kaggle/input"))

## Google Colab Environment

In [ ]:
# ============================================================
# ENVIRONMENT SETUP — GOOGLE COLAB
# Run this cell only when executing on Google Colab.
# Datasets are downloaded via the Kaggle API using kaggle.json.
# Upload your kaggle.json file to the Colab session before running.
# ============================================================
import os, sys, shutil, zipfile
from google.colab import files

# --- Kaggle credentials ---
os.makedirs("/root/.kaggle", exist_ok=True)
uploaded = files.upload()                          # upload kaggle.json when prompted
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

# --- Install dependencies ---
os.system("pip install -q kaggle torch torchvision tensorflow scikit-learn scikit-image tqdm matplotlib seaborn")

# --- Download datasets ---
os.system("kaggle datasets download -d carloscanamejoy/dataset-spines-united-v2   -p /content/data    --unzip")
os.system("kaggle datasets download -d carloscanamejoy/weights-xception-model      -p /content/weights --unzip")
os.system("kaggle datasets download -d carloscanamejoy/weights-models              -p /content/weights --unzip")
os.system("kaggle datasets download -d carloscanamejoy/weights-cvae-models         -p /content/weights --unzip")
os.system("kaggle datasets download -d carloscanamejoy/physicalmetrics             -p /content/weights --unzip")

## Load Shared Metrics

In [ ]:
# ── Load shared metrics module from Kaggle dataset ───────────────────────────
# On Kaggle: metrics.py is pre-mounted as part of the physicalmetrics dataset.
# On Colab:  metrics.py is downloaded via the Kaggle API along with the other datasets.

import importlib.util, sys, os

_METRICS_KAGGLE = "/kaggle/input/datasets/carloscanamejoy/physicalmetrics/metrics.py"
_METRICS_COLAB  = "/content/weights/metrics.py"
_METRICS_LOCAL  = os.path.join("..", "utils", "metrics.py")    # running from the repo
_metrics_path   = next(p for p in (_METRICS_KAGGLE, _METRICS_COLAB, _METRICS_LOCAL)
                       if os.path.exists(p))

spec    = importlib.util.spec_from_file_location("metrics", _metrics_path)
metrics = importlib.util.module_from_spec(spec)
spec.loader.exec_module(metrics)
sys.modules["metrics"] = metrics

from metrics import (
    STRUCTURE_MAP, STRUCTURE_NAMES, STRUCTURE_COLORS, MODEL_COLORS, PARAM_NAMES,
    circular_mask, MASK,
    masked_mse, masked_bce, masked_ssim,
    cosine_similarity_pair, cosine_similarity_batch,
    magnetization, abs_magnetization, cnn_correlation,
    structure_factor, azimuthal_average, oz_fit, chi_ensemble,
    shift_image, reflect_image,
    normalize_metrics,
    save_figure, apply_figure_style,
    center_crop, get_structure_label,
)
print(f"metrics module loaded from: {_metrics_path}")

## Shared Configuration

Auto-detects the execution environment (`_ON_KAGGLE`) and defines every path used
by this notebook — the rest of the code never hardcodes paths.

In [ ]:
import json
import pickle
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Environment auto-detection: single source of truth for all paths ---
_ON_KAGGLE = os.path.exists("/kaggle/input")

DATASET_PATH  = ("/kaggle/input/datasets/carloscanamejoy/dataset-spines-united-v2/dataset_unificado_v2.npz"
                 if _ON_KAGGLE else "/content/data/dataset_unificado_v2.npz")
XCEPTION_PATH = ("/kaggle/input/datasets/carloscanamejoy/weights-xception-model/modelo_xception_fulldatabaseV3100.h5"
                 if _ON_KAGGLE else "/content/weights/modelo_xception_fulldatabaseV3100.h5")
DDPM_PATH     = ("/kaggle/input/datasets/carloscanamejoy/weights-models/ddpm_spines_final_39crop.pt"
                 if _ON_KAGGLE else "/content/weights/ddpm_spines_final_39crop.pt")
CVAEXPN_PATH  = ("/kaggle/input/datasets/carloscanamejoy/weights-cvae-models/cvae_xception.h5"
                 if _ON_KAGGLE else "/content/weights/cvae_xception.h5")
CVAEVIT_PATH  = ("/kaggle/input/datasets/carloscanamejoy/weights-cvae-models/cvae_vit.h5"
                 if _ON_KAGGLE else "/content/weights/cvae_vit.h5")
METRICS_PATH  = ("/kaggle/input/datasets/carloscanamejoy/physicalmetrics/metrics.py"
                 if _ON_KAGGLE else "/content/weights/metrics.py")
OUTPUT_DIR    = "/kaggle/working/outputs" if _ON_KAGGLE else "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Writable checkpoint directory (Kaggle: /kaggle/working, Colab: /content)
CKPT_DIR  = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)
BEST_CKPT = os.path.join(CKPT_DIR, "cvae_xception_best.h5")

# TensorFlow: allow GPU memory growth (prevents TF from claiming all VRAM)
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

FIG_DIR = os.path.join(OUTPUT_DIR, "cvae_xception_training")
os.makedirs(FIG_DIR, exist_ok=True)

# --- Training hyperparameters ---
Z_DIM            = 128     # latent dimension (posterior and prior)
COND_EMB_DIM     = 32      # condition embedding used by the prior network
DEC_EMB_DIM      = 64      # parameter embedding concatenated at the decoder input (z + emb = 192)
EPOCHS           = 40
BATCH_SIZE       = 64
LR               = 1e-4
BETA_START       = 1e-6
BETA_MAX         = 0.18
BETA_RAMP_EPOCHS = 12
L1_WEIGHT        = 0.05
KL_ACTIVE_THRESHOLD = 0.01   # a latent unit is "active" if its mean KLD exceeds this

apply_figure_style()
print(f"TensorFlow {tf.__version__} | GPUs: {tf.config.list_physical_devices('GPU')}")

## Section 1 — Data Loading & Stratified Split

70/15/15 train/val/test split, `SEED=42`, stratified by the magnetic structure label.
Conditioning parameters θ are MinMax-normalized with a scaler fitted on the
training split only; the scaler is saved so that downstream evaluation notebooks
can condition the model identically.

In [ ]:
data = np.load(DATASET_PATH)
print(f"npz keys: {data.files}")

imgs     = data["img"].astype(np.float32)            # (N, 39, 39, 1), range [-1, 1]
params   = data["params"].astype(np.float32)         # (N, 8)
clusters = data["labels"].astype(int)                # numeric cluster ids (see STRUCTURE_MAP)
print(f"images {imgs.shape}  range [{imgs.min():.2f}, {imgs.max():.2f}] | "
      f"params {params.shape} | clusters {sorted(np.unique(clusters).tolist())}")

if imgs.ndim == 3:
    imgs = imgs[..., np.newaxis]
N = len(imgs)

# 70/15/15 split, SEED=42, stratified by cluster label
all_idx = np.arange(N)
idx_train, idx_temp = train_test_split(all_idx, test_size=0.30, random_state=SEED,
                                       stratify=clusters)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=SEED,
                                     stratify=clusters[idx_temp])
print(f"train={len(idx_train):,}  val={len(idx_val):,}  test={len(idx_test):,}")

param_scaler = MinMaxScaler().fit(params[idx_train])
with open(os.path.join(OUTPUT_DIR, "cvae_xception_param_scaler.pkl"), "wb") as f:
    pickle.dump(param_scaler, f)

def make_tf_dataset(idx, shuffle):
    x = imgs[idx]
    y = param_scaler.transform(params[idx]).astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(20_000, seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

ds_train = make_tf_dataset(idx_train, shuffle=True)
ds_val   = make_tf_dataset(idx_val, shuffle=False)

## Section 2 — Model Definition

The encoder resizes the 39×39 grayscale input to 224×224 RGB inside the graph,
so the dataset pipeline stays lightweight. Blocks 1–10 of Xception are frozen;
blocks 11–14 remain trainable.

In [ ]:
from tensorflow.keras import layers

def build_encoder_backbone():
    inp = layers.Input(shape=(39, 39, 1), name="image_39")
    x = layers.Resizing(224, 224, interpolation="bilinear")(inp)
    x = layers.Concatenate()([x, x, x])                       # grayscale → RGB, stays in [-1, 1]
    base = tf.keras.applications.Xception(weights="imagenet", include_top=False,
                                          input_tensor=x)
    trainable_prefixes = ("block11", "block12", "block13", "block14")
    n_trainable = 0
    for layer in base.layers:
        layer.trainable = layer.name.startswith(trainable_prefixes)
        n_trainable += int(layer.trainable)
    print(f"Xception backbone: {n_trainable}/{len(base.layers)} trainable layers")
    feat = layers.GlobalAveragePooling2D(name="gap")(base.output)   # (B, 2048)
    return tf.keras.Model(inp, feat, name="xception_backbone")


def build_posterior(z_dim=Z_DIM, cond_emb_dim=COND_EMB_DIM):
    feat = layers.Input(shape=(2048,), name="features")
    cond = layers.Input(shape=(8,), name="theta_norm")
    c = layers.Dense(cond_emb_dim, activation="relu", name="post_cond_emb")(cond)
    h = layers.Concatenate()([feat, c])
    h = layers.Dense(512, activation="relu")(h)
    h = layers.Dense(256, activation="relu")(h)
    mu      = layers.Dense(z_dim, name="mu_q")(h)
    log_var = layers.Dense(z_dim, name="log_var_q")(h)
    return tf.keras.Model([feat, cond], [mu, log_var], name="posterior_q")


def build_prior(z_dim=Z_DIM, cond_emb_dim=COND_EMB_DIM):
    cond = layers.Input(shape=(8,), name="theta_norm")
    c = layers.Dense(cond_emb_dim, activation="relu", name="prior_cond_emb")(cond)
    h = layers.Dense(64, activation="relu")(c)
    h = layers.Dense(128, activation="relu")(h)
    mu      = layers.Dense(z_dim, name="mu_p")(h)
    log_var = layers.Dense(z_dim, name="log_var_p")(h)
    return tf.keras.Model(cond, [mu, log_var], name="prior_p")


def _res_block(x, ch):
    h = layers.Conv2D(ch, 3, padding="same", activation="relu")(x)
    h = layers.Conv2D(ch, 3, padding="same")(h)
    return layers.Activation("relu")(layers.Add()([x, h]))


def build_decoder(z_dim=Z_DIM, dec_emb_dim=DEC_EMB_DIM):
    z    = layers.Input(shape=(z_dim,), name="z")
    cond = layers.Input(shape=(8,), name="theta_norm")
    c = layers.Dense(dec_emb_dim, activation="relu", name="dec_cond_emb")(cond)
    h = layers.Concatenate()([z, c])                          # 128 + 64 = 192
    h = layers.Dense(5 * 5 * 256, activation="relu")(h)
    h = layers.Reshape((5, 5, 256))(h)
    h = layers.Conv2DTranspose(128, 4, strides=2, padding="same", activation="relu")(h)  # 10×10
    h = layers.Conv2DTranspose(64, 4, strides=2, padding="same", activation="relu")(h)   # 20×20
    h = layers.Conv2DTranspose(32, 4, strides=2, padding="same", activation="relu")(h)   # 40×40
    h = _res_block(h, 32)
    h = _res_block(h, 32)
    h = layers.Conv2D(1, 3, padding="same", activation="tanh")(h)                        # [-1, 1]
    out = layers.Cropping2D(((0, 1), (0, 1)), name="crop_39")(h)                         # 39×39
    return tf.keras.Model([z, cond], out, name="decoder")

In [ ]:
class CVAE(tf.keras.Model):
    """Conditional VAE with learned conditional prior p(z|θ)."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.backbone  = build_encoder_backbone()
        self.posterior = build_posterior()
        self.prior     = build_prior()
        self.decoder   = build_decoder()
        self.beta = tf.Variable(BETA_START, trainable=False, dtype=tf.float32, name="beta")
        names = ["loss", "ssim", "l1", "mse", "kld_raw", "beta_kld", "var_q", "kl_active"]
        self.trackers = {n: tf.keras.metrics.Mean(name=n) for n in names}

    @property
    def metrics(self):
        return list(self.trackers.values())

    def encode(self, x, y):
        feat = self.backbone(x, training=False)
        return self.posterior([feat, y])

    def call(self, inputs, training=False):
        x, y = inputs
        feat = self.backbone(x, training=training)
        mu_q, log_var_q = self.posterior([feat, y], training=training)
        eps = tf.random.normal(tf.shape(mu_q))
        z = mu_q + tf.exp(0.5 * log_var_q) * eps
        return self.decoder([z, y], training=training)

    def compute_losses(self, x, y, training):
        feat = self.backbone(x, training=training)
        mu_q, log_var_q = self.posterior([feat, y], training=training)
        mu_p, log_var_p = self.prior(y, training=training)

        eps = tf.random.normal(tf.shape(mu_q))
        z = mu_q + tf.exp(0.5 * log_var_q) * eps
        x_hat = self.decoder([z, y], training=training)

        x01, xh01 = (x + 1.0) / 2.0, (x_hat + 1.0) / 2.0
        ssim_val  = tf.reduce_mean(tf.image.ssim(x01, xh01, max_val=1.0))
        l1  = tf.reduce_mean(tf.abs(x - x_hat))
        mse = tf.reduce_mean(tf.square(x - x_hat))

        # KLD between two diagonal Gaussians q(z|x,θ) and p(z|θ), per latent unit
        kld_units = 0.5 * (log_var_p - log_var_q
                           + (tf.exp(log_var_q) + tf.square(mu_q - mu_p)) / tf.exp(log_var_p)
                           - 1.0)                                       # (B, Z)
        kld_per_unit = tf.reduce_mean(kld_units, axis=0)                # (Z,)
        kld = tf.reduce_sum(kld_per_unit)
        kl_active = tf.reduce_mean(tf.cast(kld_per_unit > KL_ACTIVE_THRESHOLD, tf.float32))

        loss = (1.0 - ssim_val) + L1_WEIGHT * l1 + self.beta * kld
        return {
            "loss": loss, "ssim": ssim_val, "l1": l1, "mse": mse,
            "kld_raw": kld, "beta_kld": self.beta * kld,
            "var_q": tf.reduce_mean(tf.exp(log_var_q)), "kl_active": kl_active,
        }

    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            out = self.compute_losses(x, y, training=True)
        grads = tape.gradient(out["loss"], self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        for k, m in self.trackers.items():
            m.update_state(out[k])
        logs = {k: m.result() for k, m in self.trackers.items()}
        logs["kl_weight"] = self.beta
        return logs

    def test_step(self, data):
        x, y = data
        out = self.compute_losses(x, y, training=False)
        for k, m in self.trackers.items():
            m.update_state(out[k])
        return {k: m.result() for k, m in self.trackers.items()}

    def sample_from_prior(self, y_norm, n_samples=1):
        """Generate n_samples images per condition by sampling z ~ p(z|θ)."""
        y = tf.repeat(tf.convert_to_tensor(y_norm, tf.float32), n_samples, axis=0)
        mu_p, log_var_p = self.prior(y, training=False)
        z = mu_p + tf.exp(0.5 * log_var_p) * tf.random.normal(tf.shape(mu_p))
        x = self.decoder([z, y], training=False)
        return tf.reshape(x, (len(y_norm), n_samples, 39, 39)).numpy()


cvae = CVAE(name="cvae_xception")
_ = cvae((imgs[idx_val[:2]], param_scaler.transform(params[idx_val[:2]]).astype(np.float32)))
cvae.compile(optimizer=tf.keras.optimizers.Adam(LR))
print(f"Total parameters: {cvae.count_params():,}")

## Section 3 — Callbacks: β schedule, checkpoints, reconstruction snapshots

β ramps linearly from 1e-6 to 0.18 over the first 12 epochs and stays constant
afterwards (anti-posterior-collapse warmup). Reconstruction snapshots of 5 fixed
validation images are saved at epochs 1, 10, 20 and 40.

In [ ]:
def beta_for_epoch(epoch, beta_start=BETA_START, beta_max=BETA_MAX,
                   ramp_epochs=BETA_RAMP_EPOCHS):
    """epoch is 1-based. Linear ramp over ramp_epochs, then constant plateau."""
    if epoch <= ramp_epochs:
        p = (epoch - 1) / max(1, ramp_epochs - 1)
        return beta_start + (beta_max - beta_start) * p
    return beta_max


class BetaScheduler(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self.model.beta.assign(beta_for_epoch(epoch + 1))


SNAPSHOT_EPOCHS = {1, 10, 20, 40}
rng = np.random.RandomState(SEED)
snap_idx = rng.choice(idx_val, 5, replace=False)
snap_x = imgs[snap_idx]
snap_y = param_scaler.transform(params[snap_idx]).astype(np.float32)


class ReconSnapshot(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        e = epoch + 1
        if e not in SNAPSHOT_EPOCHS:
            return
        x_hat = self.model((snap_x, snap_y), training=False).numpy()
        fig, axes = plt.subplots(5, 3, figsize=(7, 11))
        for i in range(5):
            for j, (img, title) in enumerate([
                    (snap_x[i, :, :, 0], "Original"),
                    (x_hat[i, :, :, 0], "Reconstruction"),
                    (np.abs(snap_x[i, :, :, 0] - x_hat[i, :, :, 0]), "Difference")]):
                ax = axes[i, j]
                cmap, vmin, vmax = ("jet", -1, 1) if j < 2 else ("hot", 0, 1)
                ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
                ax.set_xticks([]); ax.set_yticks([])
                if i == 0:
                    ax.set_title(title)
        fig.suptitle(f"CVAE-Xception reconstructions — epoch {e}")
        save_figure(fig, os.path.join(FIG_DIR, f"reconstructions_epoch_{e:02d}"))
        plt.close(fig)


ckpt_best = tf.keras.callbacks.ModelCheckpoint(
    BEST_CKPT,
    monitor="val_loss", save_best_only=True, save_weights_only=True, verbose=1)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True, verbose=1)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", patience=5, factor=0.5, verbose=1)

## Section 4 — Training (40 epochs, batch 64)

In [ ]:
history = cvae.fit(
    ds_train,
    validation_data=ds_val,
    epochs=EPOCHS,
    callbacks=[BetaScheduler(), ReconSnapshot(), ckpt_best, early_stop, reduce_lr],
    verbose=1,
)

# Persist best weights under both required filenames and the history as JSON.
cvae.load_weights(BEST_CKPT)
cvae.save_weights(os.path.join(CKPT_DIR, "cvae_xception_best_weights.h5"))
hist = {k: [float(v) for v in vals] for k, vals in history.history.items()}
with open(os.path.join(OUTPUT_DIR, "cvae_xception_history.json"), "w") as f:
    json.dump(hist, f, indent=2)
print(f"Saved: {BEST_CKPT}, cvae_xception_best_weights.h5, cvae_xception_history.json")

## Section 5 — Training Curves & β Schedule

In [ ]:
epochs_ax = np.arange(1, len(hist["loss"]) + 1)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
panels = [
    ("Total composite loss", [("loss", "train"), ("val_loss", "val")]),
    ("SSIM (reconstruction quality)", [("ssim", "train"), ("val_ssim", "val")]),
    ("L1 reconstruction error", [("l1", "train"), ("val_l1", "val")]),
    ("KLD terms", [("kld_raw", "raw KLD (train)"), ("beta_kld", "β·KLD (train)")]),
]
for ax, (title, series) in zip(axes.flat, panels):
    for key, lbl in series:
        if key in hist:
            ax.plot(epochs_ax, hist[key], label=lbl)
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend()
fig.suptitle("CVAE-Xception — training curves (loss components per epoch)")
save_figure(fig, os.path.join(FIG_DIR, "training_curves"))
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].plot(epochs_ax, [beta_for_epoch(e) for e in epochs_ax], color="#4C72B0")
axes[0].set_title("β-KLD schedule (ramp 12 epochs → plateau 0.18)")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("β")
axes[1].plot(epochs_ax, hist["kl_active"], label="train")
if "val_kl_active" in hist:
    axes[1].plot(epochs_ax, hist["val_kl_active"], label="val")
axes[1].set_title("Fraction of active latent units (KLD > 0.01)")
axes[1].set_xlabel("Epoch"); axes[1].legend()
axes[2].plot(epochs_ax, hist["var_q"], label="train")
if "val_var_q" in hist:
    axes[2].plot(epochs_ax, hist["val_var_q"], label="val")
axes[2].set_title("Mean posterior variance var_q")
axes[2].set_xlabel("Epoch"); axes[2].legend()
fig.suptitle("CVAE-Xception — latent-space diagnostics")
save_figure(fig, os.path.join(FIG_DIR, "beta_schedule_and_latent_diagnostics"))
plt.show()

## Section 6 — Post-training Sanity Check: Prior Sampling

Sample 4 images from the learned prior p(z|θ) for 4 random test conditions and
compare against the corresponding reference image.

In [ ]:
rng = np.random.RandomState(7)
test_pick = rng.choice(idx_test, 4, replace=False)
theta_norm = param_scaler.transform(params[test_pick]).astype(np.float32)
gen = cvae.sample_from_prior(theta_norm, n_samples=4)        # (4, 4, 39, 39)

fig, axes = plt.subplots(4, 5, figsize=(11, 9))
for i in range(4):
    axes[i, 0].imshow(imgs[test_pick[i], :, :, 0], cmap="jet", vmin=-1, vmax=1)
    axes[i, 0].set_ylabel(f"θ #{test_pick[i]}")
    for k in range(4):
        axes[i, k + 1].imshow(gen[i, k], cmap="jet", vmin=-1, vmax=1)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
for j, t in enumerate(["Reference"] + [f"Sample {k+1}" for k in range(4)]):
    axes[0, j].set_title(t)
fig.suptitle("CVAE-Xception — samples from the conditional prior p(z|θ)")
save_figure(fig, os.path.join(FIG_DIR, "prior_sampling_sanity"))
plt.show()

## After Training — Upload Weights to Kaggle

After training completes, the best checkpoint is saved at:
- Kaggle: `/kaggle/working/outputs/checkpoints/cvae_xception_best.h5`
- Colab:  `/content/outputs/checkpoints/cvae_xception_best.h5`

Upload this file to the Kaggle dataset `carloscanamejoy/weights-cvae-models` before
running `generative_comparison.ipynb` or `ciclo_completo_v2.ipynb`.

> **Note**: the evaluation notebooks expect the file inside that dataset to be named
> `cvae_xception.h5` (see `CVAEXPN_PATH`) — rename `cvae_xception_best.h5` to
> `cvae_xception.h5` when uploading.